# Comparing Resampling Methods and Fairness

## Phase 3 Exploratory Data Analysis

### Research question

How do different resampling methods affect predictive performance and fairness across demographic groups in an imbalanced classification problem?

This project uses the Adult Income dataset and a decision tree classifier to compare four conditions: no resampling (the control), random oversampling, SMOTE-NC, and random undersampling. We will examine accuracy, precision, recall, and F1, along with differences in model outcomes between demographic groups. This exploratory analysis will examine data quality, class and demographic group counts, and relationships among variables to prepare for Phase 4 modeling.

### Outcome and protected groups

The target is income, and the favorable outcome is **income >50K**. The other outcome is **income <=50K**.

Sex is the protected attribute, with **male designated privileged** and **female designated unprivileged** for fairness evaluation. These definitions use the dataset's recorded `Male` and `Female` categories. We will report female-minus-male gaps for statistical parity and equal opportunity during model evaluation.

Underrepresentation describes the number of examples in a subgroup; unprivileged describes a demographic group used as the disadvantaged reference in fairness evaluation. We will examine four sex/income subgroups:

- Female with income <=50K
- Female with income >50K
- Male with income <=50K
- Male with income >50K

Sampling targets will depend on these subgroups' counts in each training fold, not on privileged or unprivileged status. Either demographic group may have subgroups resampled; female status alone does not determine underrepresentation.

### Dataset
The dataset was downloaded from https://archive.ics.uci.edu/dataset/2/adult. This is the same dataset used by our chosen research article: Zhou, Kantarcioglu, & Clifton (2023), "On Improving Fairness of AI Models with Synthetic Minority Oversampling Techniques"


### Inspect the raw files

Before preparing the data, we inspect the files directly and read the supplied documentation to identify the field names and loading requirements.


In [1]:
from pathlib import Path

DATA_DIR = Path("data")
for filename in ["adult.data", "adult.test"]:
    print(f"\n{filename}: first three lines")
    with (DATA_DIR / filename).open() as file:
        for _ in range(3):
            print(file.readline().rstrip())

print("\nadult.names: dataset documentation")
print((DATA_DIR / "adult.names").read_text())



adult.data: first three lines
39, State-gov, 77516, Bachelors, 13, Never-married, Adm-clerical, Not-in-family, White, Male, 2174, 0, 40, United-States, <=50K
50, Self-emp-not-inc, 83311, Bachelors, 13, Married-civ-spouse, Exec-managerial, Husband, White, Male, 0, 0, 13, United-States, <=50K
38, Private, 215646, HS-grad, 9, Divorced, Handlers-cleaners, Not-in-family, White, Male, 0, 0, 40, United-States, <=50K

adult.test: first three lines
|1x3 Cross validator
25, Private, 226802, 11th, 7, Never-married, Machine-op-inspct, Own-child, Black, Male, 0, 0, 40, United-States, <=50K.
38, Private, 89814, HS-grad, 9, Married-civ-spouse, Farming-fishing, Husband, White, Male, 0, 0, 50, United-States, <=50K.

adult.names: dataset documentation
| This data was extracted from the census bureau database found at
| http://www.census.gov/ftp/pub/DES/www/welcome.html
| Donor: Ronny Kohavi and Barry Becker,
|        Data Mining and Visualization
|        Silicon Graphics.
|        e-mail: ronnyk@sgi.c

### Load and inspect the observations

Using the field order in `adult.names`, we assign column names because the data files have no header. We skip the first metadata line in `adult.test` and ignore spaces immediately after delimiters. We keep the two source DataFrames for separate inspection and combine them for an overall view. The documentation says that `Unknown` values were converted to "?". At this stage, we leave `?` and the income labels as loaded so that the inspection shows their original values.


In [2]:
import pandas as pd

columns = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "native_country",
    "income"
]

adult_train = pd.read_csv(
    "data/adult.data",
    header=None,
    names=columns,
    skipinitialspace=True
)

adult_test = pd.read_csv(
    "data/adult.test",
    header=None,
    names=columns,
    skipinitialspace=True,
    skiprows=1
)

df = pd.concat(
    [adult_train, adult_test],
    ignore_index=True
)

In [3]:
for name, data in [("Training", adult_train), ("Test", adult_test), ("Combined", df)]:
    print(f"\n{name} data: {data.shape[0]:,} rows, {data.shape[1]} columns")
    display(data.head())

print("\nCombined dataset structure")
df.info()

display(pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "unique_values_including_missing": df.nunique(dropna=False),
    "missing_values_recognized_by_pandas": df.isna().sum(),
    "question_mark_values": df.eq("?").sum()
}))



Training data: 32,561 rows, 15 columns


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K



Test data: 16,281 rows, 15 columns


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K.
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K.
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K.
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K.
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K.



Combined data: 48,842 rows, 15 columns


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K



Combined dataset structure
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             48842 non-null  int64 
 1   workclass       48842 non-null  object
 2   fnlwgt          48842 non-null  int64 
 3   education       48842 non-null  object
 4   education_num   48842 non-null  int64 
 5   marital_status  48842 non-null  object
 6   occupation      48842 non-null  object
 7   relationship    48842 non-null  object
 8   race            48842 non-null  object
 9   sex             48842 non-null  object
 10  capital_gain    48842 non-null  int64 
 11  capital_loss    48842 non-null  int64 
 12  hours_per_week  48842 non-null  int64 
 13  native_country  48842 non-null  object
 14  income          48842 non-null  object
dtypes: int64(6), object(9)
memory usage: 5.6+ MB


,dtype,unique_values_including_missing,missing_values_recognized_by_pandas,question_mark_values
age,int64,74,0,0
workclass,object,9,0,2799
fnlwgt,int64,28523,0,0
education,object,16,0,0
education_num,int64,16,0,0
marital_status,object,7,0,0
occupation,object,15,0,2809
relationship,object,6,0,0
race,object,5,0,0
sex,object,2,0,0


### Inspect the values in each column

We display every distinct categorical value and its count separately for training and test data. For numerical columns, we show descriptive statistics and a sorted sample of distinct values; the unique-value counts above indicate how many distinct values exist in total.


In [5]:
for column in df.select_dtypes(include="number").columns:
    print(f"\n{column}: numerical summary")
    with pd.option_context("display.float_format", lambda value: f"{value:.3f}".rstrip("0").rstrip(".")):
        display(df[column].describe().to_frame(name=column))
    values = sorted(df[column].dropna().unique().tolist())

for column in df.select_dtypes(exclude="number").columns:
    print(f"\n{column}: all distinct values and counts by source file")
    counts = pd.concat({
        "training_count": adult_train[column].value_counts(dropna=False),
        "test_count": adult_test[column].value_counts(dropna=False)
    }, axis=1).fillna(0).astype(int)
    with pd.option_context("display.max_rows", None):
        display(counts)



age: numerical summary


,age
count,48842
mean,38.644
std,13.711
min,17
25%,28
50%,37
75%,48
max,90



fnlwgt: numerical summary


,fnlwgt
count,48842
mean,189664.135
std,105604.025
min,12285
25%,117550.5
50%,178144.5
75%,237642
max,1490400



education_num: numerical summary


...

,training_count,test_count
sex,,
Male,21790,10860
Female,10771,5421



native_country: all distinct values and counts by source file


,training_count,test_count
native_country,,
United-States,29170,14662
Mexico,643,308
?,583,274
Philippines,198,97
Germany,137,69
Canada,121,61
Puerto-Rico,114,70
El-Salvador,106,49
India,100,51



income: all distinct values and counts by source file


,training_count,test_count
income,,
<=50K,24720,0
>50K,7841,0
<=50K.,0,12435
>50K.,0,3846


### Column meanings and feature types

The following names match the Python column names used above. Underscores replace the hyphens used in the raw documentation, such as `education-num` becoming `education_num`. These are conceptual feature types; the loaded Python data types are displayed above.

| Column | Feature type | Meaning and role |
| --- | --- | --- |
| `age` | Numerical, integer | Age in years. |
| `workclass` | Categorical, nominal | Employment category, such as `Private`, `Self-emp-not-inc`, or `State-gov`. |
| `fnlwgt` | Numerical, integer | Final census sample weight. It represents how many people in the population a sampled record represents; it is not an individual's income. Its role in modeling or weighting will require an explicit decision. |
| `education` | Categorical, ordered | Highest education category, such as `HS-grad`, `Bachelors`, or `Masters`. |
| `education_num` | Numerical, ordinal code | Ordered numerical representation of education level. It overlaps with `education`; we will examine their relationship before choosing modeling inputs. |
| `marital_status` | Categorical, nominal | Marital status, such as `Never-married`, `Married-civ-spouse`, or `Divorced`. |
| `occupation` | Categorical, nominal | Occupational category, such as `Sales`, `Craft-repair`, or `Prof-specialty`. |
| `relationship` | Categorical, nominal | Household relationship category, such as `Husband`, `Wife`, or `Own-child`. |
| `race` | Categorical, nominal | Recorded race category. Sex is the protected attribute selected for this project's fairness comparisons. |
| `sex` | Categorical, binary | Recorded `Female` or `Male` category. Protected attribute: female is designated unprivileged and male privileged. |
| `capital_gain` | Numerical, integer | Recorded capital gain amount. |
| `capital_loss` | Numerical, integer | Recorded capital loss amount. |
| `hours_per_week` | Numerical, integer | Reported hours worked per week. |
| `native_country` | Categorical, nominal | Recorded country of origin category. |
| `income` | Categorical, binary target | Income category, intended as `<=50K` or `>50K`. The raw test labels include trailing periods, as shown above; these will need normalization. Income `>50K` is the favorable outcome. |

These descriptions follow the local `adult.names` documentation.

#### Findings from the initial inspection

The displayed shapes show 32,561 training observations and 16,281 test observations, giving 48,842 observations and 15 columns in the combined data. The loaded types are six integer columns and nine object columns. The object columns contain categorical labels, including the income target; an integer storage type does not by itself mean a feature is continuous, as illustrated by the ordered `education_num` code.

**Unknown values and label consistency.** Pandas reports no recognized missing values at this stage, but the explicit `?` check finds unknown values in `workclass` (2,799), `occupation` (2,809), and `native_country` (857). These are counts of entries per column; one row may have unknown values in more than one column. The counts show why checking only `isna()` would miss true missing values. We will need to recognize these markers as missing before deciding how to handle them. The income frequency table also shows four raw strings instead of the intended two outcome categories: `<=50K` and `>50K` in training, and `<=50K.` and `>50K.` in test. The trailing periods need to be standardized before analyzing the combined outcome distribution.

**Numerical ranges and values to investigate.** The following observations come from the numerical summaries above. Extreme values are candidates for further examination.

| Feature | Observed minimum | Observed maximum | Interpretation and follow-up |
| --- | ---: | ---: | --- |
| `age` | 17 | 90 | The median is 37 and the middle half spans 28–48. Ages near 90 are far above the upper quartile and warrant a closer look at their frequency and distribution. |
| `fnlwgt` | 12,285 | 1,490,400 | The median is 178,144.5 and the upper quartile is 237,642. The large maximum suggests a long upper tail to examine. This variable is a census sample weight, so large values require interpretation in that context. |
| `education_num` | 1 | 16 | There are 16 distinct codes, matching the number of education categories. We should check their mapping before deciding whether both education columns are needed. |
| `capital_gain` | 0 | 99,999 | The first quartile, median, and third quartile are all zero, while the mean is 1,079.068. This indicates a concentration at zero with a high positive upper tail. We should examine nonzero values and the frequency of the maximum. |
| `capital_loss` | 0 | 4,356 | All three quartiles are zero, while the mean is 87.502. As with capital gain, the nonzero values need separate attention because an overall summary is dominated by zeros. |
| `hours_per_week` | 1 | 99 | The median is 40 and the middle half spans 40–45 hours. Both 1 and 99 are far from typical reported hours. |

**Categorical imbalance and rare values.** The sex frequency table contains 32,650 male and 16,192 female observations, so the two demographic groups have unequal representation. This does not yet tell us which sex/income subgroups are smallest, which would require a cross-tabulation after normalizing income labels. The categorical tables also show substantial imbalance within predictors: `Private` is the largest workclass category and `United-States` is the largest native-country category. Some categories are very rare: `Never-worked` has 10 observations, `Armed-Forces` has 15, and `Holand-Netherlands` has one training observation and none in test. Rare categories may be absent from individual folds, which will matter when preparing categorical encoding. We should retain the documented category labels unless a later, justified preparation step changes them.
